In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial

# --- Project imports ---
from GX_271.gilson_ethernet import GilsonEthernet
from GX_271.tray import Tray
from GX_271.rack import Rack_209
from GX_271.flow_logging import FlowLogger
from GX_271.dim import DIM

#--- Project imports from other folders---
from WPI_Syr_pump.Pump import AL1000

#--- Construct the tray ---
tray = Tray()

# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.103", admin_port=50185, tray=tray)

# --- Start logging (declare this run belongs to an "experiment") ---
logger = FlowLogger()
logger.start_experiment("Development")

# --- Instantiate modules (racks, wash stations, etc.) (these know internal geometry) ---
rack1 = Rack_209()  
#rack2 = Rack_209()
#DIM = DIM()

# --- Register modules on the tray with global offsets, previously handled by each module in the tray ---
tray.add_module(slot = 1, name = "rack1", module = rack1, x_offset = 35.5, y_offset = 7.2)
#tray.add_module(slot = 2, name = "rack2", module = rack2, x_offset = 165, y_offset = 7.2) # Simulates another rack - useful later
#tray.add_module(slot = 3, name = "DIM", module = DIM, x_offset = xxx, y_offset = xxx) # Add real offsets later


# --- Instantiate serial object for AL1000 pump ----
ser = serial.Serial("COM2", 9600, timeout=0.5)
pump = AL1000(ser, device_address="@00", sleep_time=0.5)

pump.connect()

syringe_dia = 40    # mm
rate = 10            # mL/min
volume = 10           # mL
direction = "INF"     # Infuse

pump.prepare(dia=syringe_dia, rate=rate, volume=volume, direction=direction)










In [ ]:
# Dont touch this - ignore for now - just want to show how you might instantiate the runze valves + connect

r3port = Runze3Port(port="COM?", baudrate=9600, valve_address=1)
r4port = Runze4Port(port="COM?", baudrate=9600, valve_address=1) # Can use the same address, each has its own wire (Not an RS-485 bus)
r3port.connect()
r4port.connect()

In [3]:
# Checking that both racks are registered and ready to go

print("Tray modules:", tray.list_modules())
print("rack1 offsets:", tray.get_offsets("rack1"))
print("rack2 offsets:", tray.get_offsets("rack2"))

Tray modules: ['rack1', 'rack2']
rack1 offsets: (35.5, 7.2)
rack2 offsets: (170.0, 7.2)


In [4]:
g.home()

All axes homed successfully and Z is within safe limits


In [5]:
g.home()   
g.go_to_vial("rack1", 1)
time.sleep(2)
g.go_to_vial("rack1", 16)
time.sleep(2)
g.go_to_vial("rack2", 1)
time.sleep(2)
g.go_to_vial("rack2", 16)


All axes homed successfully and Z is within safe limits
Moving to rack1 vial 1 at (35.50, 7.20) mm
Moving to rack1 vial 16 at (35.50, 273.75) mm
Moving to rack2 vial 1 at (170.00, 7.20) mm
Moving to rack2 vial 16 at (170.00, 273.75) mm


(np.float64(170.0), np.float64(273.75))

In [5]:
print(g.current_z)

55
